# Pipeline Overview — Architecture, Resources, Status

Live system health, disk usage, and architecture reference.

In [1]:
import os, json, subprocess, shutil

print('=== Architecture ===')
print('''
User Query
    |
    v
Claude Query Expansion (3 Greek variants)
    |
    +---> Recoll BM25 (Xapian, 6.9G index)
    |         filename: and dir: field queries
    |
    +---> FAISS x5 models (cohere/e5/gte/bge-m3)
    |         80K chunks, cosine similarity
    |
    v
vector_augment (BM25 order kept, vector-only appended)
    |
    v
5 Parallel Claude Agents -> JSON facts
    |
    v
Merge + Dedup + Chronological Sort
    |
    v
Final Answer (with source citations)
''')

=== Architecture ===

User Query
    |
    v
Claude Query Expansion (3 Greek variants)
    |
    +---> Recoll BM25 (Xapian, 6.9G index)
    |         filename: and dir: field queries
    |
    +---> FAISS x5 models (cohere/e5/gte/bge-m3)
    |         80K chunks, cosine similarity
    |
    v
vector_augment (BM25 order kept, vector-only appended)
    |
    v
5 Parallel Claude Agents -> JSON facts
    |
    v
Merge + Dedup + Chronological Sort
    |
    v
Final Answer (with source citations)



In [5]:
# Live resource inventory
import pandas as pd

resources = [
    {'Resource': 'Xapian BM25 index', 'Path': '$XAPIAN_DB/', 'Type': 'BM25'},
    {'Resource': 'FAISS cohere-v3', 'Path': '$FAISS_BASE/cohere-v3/', 'Type': 'Vector'},
    {'Resource': 'FAISS e5-base', 'Path': '$FAISS_BASE/e5-base/', 'Type': 'Vector'},
    {'Resource': 'FAISS e5-large', 'Path': '$FAISS_BASE/e5-large/', 'Type': 'Vector'},
    {'Resource': 'FAISS GTE', 'Path': '$FAISS_BASE/gte/', 'Type': 'Vector'},
    {'Resource': 'FAISS bge-m3', 'Path': '$FAISS_BASE/bge-m3/', 'Type': 'Vector'},
    {'Resource': 'ChromaDB', 'Path': '$CHROMADB_DIR/', 'Type': 'Vector'},
    {'Resource': 'Source corpus', 'Path': '$CORPUS_DIR/', 'Type': 'Data'},
    {'Resource': 'HF model cache', 'Path': '$HF_HOME/', 'Type': 'Infra'},
    {'Resource': 'Venv gte-embed', 'Path': '$VENV_GTE/', 'Type': 'Infra'},
    {'Resource': 'Venv transformers', 'Path': '$VENV_TF/', 'Type': 'Infra'},
    {'Resource': 'OCR MCP server', 'Path': '$OCR_SERVER/', 'Type': 'OCR'},
    {'Resource': 'Recoll config', 'Path': '~/.recoll/recoll.conf', 'Type': 'Config'},
    {'Resource': 'API keys', 'Path': '~/.config/dsearch/.env', 'Type': 'Config'},
]

# Check sizes
for r in resources:
    p = r['Path']
    if os.path.isdir(p):
        try:
            result = subprocess.run(['du', '-sh', p], capture_output=True, text=True, timeout=10)
            r['Size'] = result.stdout.split()[0] if result.returncode == 0 else '?'
        except:
            r['Size'] = '?'
    elif os.path.isfile(p):
        r['Size'] = f'{os.path.getsize(p)/1024:.0f}K'
    else:
        r['Size'] = 'MISSING'
    r['Exists'] = 'Yes' if os.path.exists(p) else 'NO'

df = pd.DataFrame(resources)

type_colors = {
    'BM25': '#2196F3',
    'Vector': '#4CAF50',
    'Data': '#FF9800',
    'OCR': '#FFC107',
    'Infra': '#9C27B0',
    'Config': '#607D8B',
}

def color_type(row):
    bg = type_colors.get(row['Type'], '')
    styles = [f'background-color: {bg}'] * len(row)
    if row['Exists'] == 'NO':
        styles = ['background-color: #FFCDD2'] * len(row)
    return styles

display(df.style.apply(color_type, axis=1).set_caption('Resource Inventory — Live Status'))

,Resource,Path,Type,Size,Exists
0,Xapian BM25 index,$XAPIAN_DB/,BM25,6.9G,Yes
1,FAISS cohere-v3,$FAISS_BASE/cohere-v3/,Vector,316M,Yes
2,FAISS e5-base,$FAISS_BASE/e5-base/,Vector,250M,Yes
3,FAISS e5-large,$FAISS_BASE/e5-large/,Vector,316M,Yes
4,FAISS GTE,$FAISS_BASE/gte/,Vector,250M,Yes
5,FAISS bge-m3,$FAISS_BASE/bge-m3/,Vector,316M,Yes
6,ChromaDB,$CHROMADB_DIR/,Vector,614M,Yes
7,Source corpus,$CORPUS_DIR/,Data,28G,Yes
8,HF model cache,$HF_HOME/,Infra,8.5G,Yes
9,Venv gte-embed,$VENV_GTE/,Infra,2.0G,Yes


In [6]:
# OCR coverage
corpus = '$CORPUS_DIR'
gemini = subprocess.run(f'find {corpus} -name "*.ocr.gemini.txt" 2>/dev/null | wc -l',
                       shell=True, capture_output=True, text=True)
gemini_pro = subprocess.run(f'find {corpus} -name "*.ocr.gemini-pro.txt" 2>/dev/null | wc -l',
                           shell=True, capture_output=True, text=True)
images = subprocess.run(f'find {corpus} -type f \\( -iname "*.jpg" -o -iname "*.png" -o -iname "*.jpeg" \\) 2>/dev/null | wc -l',
                       shell=True, capture_output=True, text=True)

n_gemini = int(gemini.stdout.strip())
n_pro = int(gemini_pro.stdout.strip())
n_images = int(images.stdout.strip())
n_ocr = n_gemini + n_pro

print(f'OCR Coverage:')
print(f'  Total images:     {n_images:,}')
print(f'  OCR cached:       {n_ocr:,} ({n_ocr/max(n_images,1)*100:.0f}%)')
print(f'    gemini:         {n_gemini:,}')
print(f'    gemini-pro:     {n_pro:,}')
print(f'  Remaining:        {n_images - n_ocr:,}')

OCR Coverage:
  Total images:     8,373
  OCR cached:       4,863 (58%)
    gemini:         2,403
    gemini-pro:     2,460
  Remaining:        3,510


In [7]:
# CLI tools check
cli_tools = [
    ('/usr/local/bin/dsearch-multimodel', 'Multi-model vector search (Cohere default)'),
    ('/usr/local/bin/docsearch', 'Hybrid BM25+vector sample search'),
    ('/usr/local/bin/docsearch-all', 'Full-disk BM25+Claude'),
]

print('CLI Tools:')
for path, desc in cli_tools:
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{exists}] {path} — {desc}')

CLI Tools:
  [OK] /usr/local/bin/dsearch-multimodel — Multi-model vector search (Cohere default)
  [OK] /usr/local/bin/docsearch — Hybrid BM25+vector sample search
  [OK] /usr/local/bin/docsearch-all — Full-disk BM25+Claude


## Known Gaps

- [ ] Extensionless text files not indexed (30 files in corpus)
- [ ] BM25 not in 6-step pipeline as 6th source
- [ ] Doc count discrepancy (5,707 vs 8,304 across models)
- [ ] Full-disk hybrid benchmark not run (only 13q sample)
- [ ] OCR coverage incomplete (Health, Finance, RealEstate)
- [ ] Image-only PDFs skipped (~8K affected)
- [ ] Query expansion can hurt (generic English terms)